# 教程 06：时序类数据预测 — TimesNet

**学习目标** ⭐
- 了解 TimesNet 的核心理念
- 学习 FFT 周期发现和 1D→2D 变形
- 使用本仓库的 TimesNet 参考实现进行预测

---

## TimesNet 简介

TimesNet (Wu et al., ICLR 2023) 是一个通用的时序建模框架，核心思想：

1. **1D → 2D 变形**：用 FFT 自动发现时序中的周期，将 1D 序列折叠为 2D 张量
2. **2D 卷积捕获时序模式**：在 2D 空间中用 Inception 块同时捕获周期内和周期间的依赖
3. **自适应聚合**：按频率幅度加权融合多个周期的结果

```
输入 1D 序列 ──→ FFT 发现周期 ──→ 折叠为 2D ──→ 2D 卷积 ──→ 展开回 1D ──→ 输出
```

这种方法将时序分析的问题转化为计算机视觉中的 2D 图像分析问题，充分利用了 2D 卷积的强大表达能力。

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, repo_root)

try:
    from prediction.TimeSeries.timesnet import (
        TimesNet, TimesBlock, fft_for_period
    )
    import torch
    import numpy as np
    print("TimesNet 模块加载成功 ✅")
except ImportError as e:
    print(f"加载失败: {e}")

## 生成合成时序数据

生成包含多个周期分量的合成时间序列，验证 TimesNet 能否自动发现周期。

In [ ]:
import numpy as np
import torch

np.random.seed(42)
seq_len, pred_len = 192, 96
t = np.arange(seq_len + pred_len, dtype=np.float32)

# 组合多个频率的分量
data = (
    np.sin(2 * np.pi * t / 24) +       # 日周期 (周期=24)
    0.5 * np.sin(2 * np.pi * t / 12) +  # 半日周期 (周期=12)
    0.3 * np.cos(2 * np.pi * t / 48) +  # 双日周期 (周期=48)
    0.2 * np.random.randn(len(t))        # 噪声
).astype(np.float32)

data = data.reshape(1, -1, 1)
x_train = torch.from_numpy(data[:, :seq_len, :])
y_train = torch.from_numpy(data[:, seq_len:, :])

print(f"输入形状: {x_train.shape}")
print(f"目标形状: {y_train.shape}")
print(f"真实周期: 24, 12, 48")

In [ ]:
# 用 FFT 自动发现周期
periods, amplitudes = fft_for_period(x_train, k=3)
print(f"发现的周期: {periods.tolist()}")
print(f"对应的振幅: {amplitudes[0].tolist()}")

In [ ]:
model = TimesNet(
    seq_len=seq_len,
    pred_len=pred_len,
    d_model=32,
    d_ff=64,
    n_layers=2,
    top_k=3,
    num_features=1,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("训练 TimesNet...")
for epoch in range(50):
    optimizer.zero_grad()
    pred = model(x_train)
    loss = (pred - y_train).pow(2).mean()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"  Epoch {epoch}: loss = {loss.item():.6f}")
print(f"训练完成, 最终 loss = {loss.item():.6f}")

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(x_train)

try:
    import matplotlib.pyplot as plt

    full_series = data[0, :, 0]
    pred_series = pred[0, :, 0].numpy()
    true_series = y_train[0, :, 0].numpy()

    plt.figure(figsize=(14, 5))
    plt.plot(range(seq_len), full_series[:seq_len], 'b-', label='输入历史', linewidth=1)
    plt.plot(range(seq_len, seq_len + pred_len), true_series, 'g-', label='真实值', linewidth=2)
    plt.plot(range(seq_len, seq_len + pred_len), pred_series, 'r--', label='预测值', linewidth=2)
    plt.axvline(x=seq_len, color='gray', linestyle=':', alpha=0.5)
    plt.xlabel('时间步')
    plt.ylabel('值')
    plt.title('TimesNet 时序预测')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    rmse = np.sqrt(np.mean((pred_series - true_series) ** 2))
    print(f"RMSE: {rmse:.4f}")
except ImportError:
    print("需要 matplotlib")

## 总结 📋

- TimesNet 将时序分析转化为 2D 图像分析，充分利用 FFT 发现的周期模式
- 核心流程：FFT 周期发现 → 1D→2D 变形 → 2D 卷积 → 自适应聚合
- 本仓库的 `prediction/TimeSeries/timesnet.py` 提供完整 PyTorch 参考实现
- 合成验证表明，TimesNet 能准确发现时序中的多周期模式并做出预测

## 思考练习 ✏️

1. 改变合成数据中的周期和振幅，观察 TimesNet 的适应能力
2. 尝试多变量时序预测（num_features > 1）
3. 对比 TimesNet 与 LSTM 在该任务上的表现
4. 思考：TimesNet 的 2D 卷积部分如何迁移到 Ascend C 实现？